In [89]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [90]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-05-16,99.522125,99.885480,98.348206,98.441379
1,2016-05-17,98.273666,99.727086,98.012790,99.438267
2,2016-05-18,98.627708,99.158766,97.863731,98.050064
3,2016-05-19,98.115265,98.487938,97.397874,98.245700
4,2016-05-20,99.196014,99.596637,98.450667,98.506573
...,...,...,...,...,...
2509,2026-05-08,711.229980,711.229980,699.500000,699.919983
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022


In [91]:
def RSI(data, n):
    pc = data['Close'].diff()
    up = pc.clip(lower=0)
    dn = pc.clip(upper=0).abs()
    avg_up = up.ewm(alpha=1/n, adjust=False).mean()
    avg_dn = dn.ewm(alpha=1/n, adjust=False).mean()
    RS = avg_up / avg_dn
    RSI = 100 - (100 / (1 + RS))
    return RSI

df['RSI14'] = RSI(df, 28)
df['RSI3'] = RSI(df, 6)
    

In [92]:
def CMB(data):
    RSI_Chg = ((data.RSI14 - data.RSI14.iloc[-9]) / data.RSI14.iloc[-9]) * 100
    RSI_Mom = data.RSI3.rolling(3).mean()
    CIL = RSI_Chg + RSI_Mom
    Fast = CIL.rolling(13).mean()
    Slow = CIL.rolling(33).mean()
    return CIL, Fast, Slow

df['CIL'], df['Fast'], df['Slow'] = CMB(df)
df.dropna(inplace=True)
df

Price,Date,Close,High,Low,Open,RSI14,RSI3,CIL,Fast,Slow
35,2016-07-06,101.169769,101.216473,99.553666,99.871284,34.344419,60.293550,10.533119,-21.667449,-31.509479
36,2016-07-07,101.468674,101.692871,100.992249,101.263157,35.062960,62.305176,12.398458,-16.708704,-28.254396
37,2016-07-08,103.038078,103.094126,101.842345,102.010494,38.714967,71.426108,23.503051,-11.495211,-25.031042
38,2016-07-11,103.626587,104.009590,103.374366,103.383703,40.026592,74.231805,30.142175,-6.170224,-21.766067
39,2016-07-12,104.149727,104.429978,104.009601,104.261829,41.186941,76.674903,36.695256,-0.780928,-18.982235
...,...,...,...,...,...,...,...,...,...,...
2509,2026-05-08,711.229980,711.229980,699.500000,699.919983,72.668278,91.524551,99.588416,81.521825,54.590754
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985,72.992060,91.951190,101.062969,83.014687,57.752742
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971,70.450208,78.096447,94.241119,84.403816,60.462613
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022,71.711586,82.093960,93.014277,85.295758,63.429753


In [93]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data.CIL.iloc[i-1] < data.Fast.iloc[i-1]) and (data.CIL.iloc[i] > data.Fast.iloc[i]):
            signal[i] = 1
        elif (data.Fast.iloc[i-1] > data.Slow.iloc[i-1]) and (data.Fast.iloc[i] < data.Slow.iloc[i]):
            signal[i] = 2
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)


In [94]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2256
1     164
2      59
Name: count, dtype: int64


(2479, 13)

In [95]:
df.set_index('Date', inplace=True)

In [96]:
bar = 2150
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.CIL, 
                         line=dict(color='white', width=1),
                         name='CIL'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.Fast, 
                         line=dict(color='red', width=1),
                         name='Fast'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.Slow, 
                         line=dict(color='lightseagreen', width=1),
                         name='Slow'),
                         row=2, col=1)

fig.update_layout(
    annotations=[dict(text="CMB Composite Index Indicator ",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=1.00,
                    y=0.35,
                    showarrow=False)])

fig.add_hline(y=100, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=50, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=-50, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()


In [101]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.03*price, sl=0.9*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99, tp=0.95*price, sl=1.03*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Start                     2016-07-06 00:00:00
End                       2026-05-14 00:00:00
Duration                   3599 days 00:00:00
Exposure Time [%]                     56.5954
Equity Final [$]                 702027.63023
Equity Peak [$]                  702027.63023
Commissions [$]                   55055.67345
Return [%]                          602.02763
Buy & Hold Return [%]               611.46745
Return (Ann.) [%]                    21.90884
Volatility (Ann.) [%]                17.14412
CAGR [%]                             14.62022
Sharpe Ratio                          1.27792
Sortino Ratio                         2.42857
Calmar Ratio                          1.15096
Alpha [%]                            516.2491
Beta                                  0.14028
Max. Drawdown [%]                   -19.03522
Avg. Drawdown [%]                    -2.17503
Max. Drawdown Duration      252 days 00:00:00
Avg. Drawdown Duration       19 days 00:00:00
# Trades                          

In [98]:
trades = stats['_trades']
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='CMB Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [99]:
def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='CMB Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='Equity',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()